GPU được kiểm tra sau khi cài dependencies pinned, bằng tiến trình Python mới ở cell smoke.
Chạy notebook từ đầu để mount Drive và cập nhật repo trước khi chạy experiment.


# 04D — Portable local / Colab setup and experiments

Default: **non-training smoke test**. Locally, use the current checkout and a pinned Python environment; set `EDGEAI_DATA_ROOT` before importing tools to use an external EdgeAI storage tree. Colab mounts Drive, clones or fast-forward pulls the GitHub repository, and installs its pinned runtime requirements. Code executes in the connected kernel's filesystem.

Upload the complete staged `EdgeAI` tree to `/content/drive/MyDrive/EdgeAI` and commit/push local code fixes before the first Colab run. Never run 00–03B to repair a missing cache. This notebook does not push code or upload data.


In [ ]:
from pathlib import Path, PureWindowsPath
import os, subprocess, sys
try:
    from google.colab import drive
except ImportError:
    drive = None
IN_COLAB = drive is not None

REPO_URL = "https://github.com/ancaranovik/Mosquito_Wingbeat_v1.git"
if IN_COLAB:
    drive.mount("/content/drive")
    assert REPO_URL.startswith("https://github.com/") and "@" not in REPO_URL
    repo_hint = os.environ.get("EDGEAI_REPO_ROOT", "/content/edge-ai")
    if PureWindowsPath(repo_hint).drive:
        raise ValueError("Colab needs a runtime checkout, not a Windows repository path")
    repo = Path(repo_hint).resolve()
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    else:
        origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=repo, text=True).strip()
        assert origin == REPO_URL, "Existing checkout has a different origin"
        assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip(), "Preserve uncommitted runtime changes before pulling"
        subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
else:
    hint = Path(os.environ.get("EDGEAI_REPO_ROOT", Path.cwd())).resolve()
    repo = next((p for p in [hint, *hint.parents] if (p / "tools/project_paths.py").is_file()), None)
    if repo is None:
        raise RuntimeError("Set EDGEAI_REPO_ROOT to your local checkout")
os.environ["EDGEAI_REPO_ROOT"] = str(repo)
os.chdir(repo)
sys.path.insert(0, str(repo / "tools"))
sys.dont_write_bytecode = True
print("Mode:", "COLAB" if IN_COLAB else "LOCAL")
print("Checkout:", repo)


In Colab, install the pinned cache-consumer and notebook dependencies. Locally, use the already configured environment without installing packages. Verification and smoke testing use fresh child Python processes. Configure private GitHub authentication interactively; never embed credentials here.


In [ ]:
from colab_bootstrap import configure
paths = configure(install=IN_COLAB)


Verify all frozen identities and eight saved baseline runs. This reads saved predictions and hashes; it does not evaluate a trained model on TEST again or regenerate any data.

In [ ]:
subprocess.run([sys.executable, "-B", "tools/verify_portability.py"], cwd=repo, check=True)


Colab requires CUDA and reports the GPU name before eight synthetic forward checks. Local mode uses CPU. Neither mode constructs an optimizer or starts training.


In [ ]:
device = "cuda" if IN_COLAB else "cpu"
subprocess.run([sys.executable, "-B", "tools/experiment_runner.py", "smoke", "--device", device], cwd=repo, check=True)
print("Future outputs:", paths.EXPERIMENT_ROOT / "<experiment_id>")


## Lần tiếp theo: exp_002_control_v2 — một experiment gồm 8 cấu hình

Chỉnh tên và config trong cell ngay bên dưới. `control_v2.json` chọn checkpoint
có validation Macro-F1 cao nhất, đồng thời lưu checkpoint có validation loss thấp nhất.
Learning rate 0.001, seed 42, class weights, kiến trúc và cache giữ theo baseline.
Early stopping vẫn theo validation loss (patience 8), để lần đối chứng chỉ đổi cách chọn checkpoint.
TRAIN ở chế độ eval và validation predictions được lưu để chẩn đoán; TEST mặc định tắt khi tuning.

Sau khi verify và GPU smoke PASS, đổi `RUN_TRAINING` thành `True` trong bản điều khiển của bạn.
Giữ `False` trong notebook commit/push. Nếu notebook nằm ngay trong checkout runtime,
sao chép nó ra ngoài checkout trước khi chỉnh điều khiển; runner yêu cầu Git sạch.
Nếu dùng VS Code local + kernel Colab, chỉnh bản local không sửa checkout runtime.

Kết quả lưu vào `MyDrive/EdgeAI/experiments/<EXPERIMENT_NAME>/`, chạy lần lượt
4 model với 03A rồi 4 model với 03B. Tên đã tồn tại bị từ chối, kể cả lần FAILED/INTERRUPTED.
Lần thử lại cần tên mới. 04C đọc đúng tên này, không cần train trong 04A/04B hoặc tạo lại 03A/03B.

Đọc hướng dẫn: `docs/EXPERIMENT_V2_GUIDE.md`. Các config LR/normalization đã chuẩn bị
cho bước sau; chạy và xem exp_002 trước khi quyết định bước tiếp theo.


In [ ]:
# CHỈNH ĐIỀU KHIỂN TẠI ĐÂY; mỗi lần train dùng tên chưa tồn tại.
EXPERIMENT_NAME = "exp_002_control_v2"
EXPERIMENT_DESCRIPTION = "Lần 2 đối chứng: chọn checkpoint theo validation Macro-F1; giữ LR=0.001, no normalization và early stopping theo loss. TRAIN eval + validation diagnostics; TEST tắt."
EXPERIMENT_CONFIG = "configs/experiments/control_v2.json"
RUN_TRAINING = False

from cache_consumer import read_json
from experiment_runner import run_plan, training_module
config_preview = read_json(repo / EXPERIMENT_CONFIG)
config_preview.update(experiment_id=EXPERIMENT_NAME, description=EXPERIMENT_DESCRIPTION)
plan = run_plan(config_preview)
engine = training_module(config_preview)
effective = (engine.effective_config(config_preview["training_overrides"], config_preview["evaluate_test"])
             if "protocol_version" in config_preview else engine.CONFIG)
experiment_path = paths.experiment(EXPERIMENT_NAME)
print("Experiment:", EXPERIMENT_NAME)
print("Kết quả:", experiment_path)
print("Reference:", config_preview.get("reference_experiment_id", "accepted baseline"))
print("Params thực thi:", effective)
print("Kế hoạch:", len(plan), "cấu hình, chạy lần lượt:", [p["run_id"] for p in plan])
if RUN_TRAINING:
    subprocess.run([
        sys.executable, "-u", "-B", "tools/experiment_runner.py", "train",
        "--config", EXPERIMENT_CONFIG, "--experiment-id", EXPERIMENT_NAME,
        "--description", EXPERIMENT_DESCRIPTION,
    ], cwd=repo, check=True)
else:
    print("Training disabled. Kiểm tra tên/config/params, rồi đổi RUN_TRAINING thành True khi muốn chạy.")


## Xem kết quả sau khi đủ 8 cấu hình hoàn tất

Cell này chỉ đọc/xác thực kết quả; không train hoặc chạy lại TEST. Trong 04C, đặt cùng `EXPERIMENT_NAME` để xem history và confusion matrix của cả 8 cấu hình.


In [ ]:
if RUN_TRAINING:
    from experiment_review import compare_saved_results
    from IPython.display import display
    display(compare_saved_results(EXPERIMENT_NAME))
else:
    print("Để xem lần đã chạy, mở 04C và nhập EXPERIMENT_NAME tương ứng; không cần bật training.")
